#Mini Trabalho DAA

##Ano Letivo 25/26

###Trabalho Realizado por:


*   Daniel Silva, nº 129859
*   Francisco Silva, nº 129868
*   Rodrigo Cruz, nº 129829

#Análise de Redes

A análise de redes constitui uma área central da teoria dos grafos, com aplicações que vão desde redes sociais e infraestruturas até sistemas biológicos. Neste trabalho, no âmbito da unidade curricular de Desenho e Análise de Algoritmos (DAA), aplicamos estas técnicas ao estudo de universos ficcionais — concretamente, às redes de co-ocorrências de personagens da saga A Song of Ice and Fire (Game of Thrones) e do universo Marvel Comics.
O objetivo principal foi implementar, de raiz em Python, um conjunto de métricas estatísticas para análise de grafos não orientados, organizadas numa API denominada CentralityAnalyzer. Esta implementação foi desenvolvida sem recurso a bibliotecas externas de análise de grafos (como networkx ou igraph), partindo da estrutura de dados Graph desenvolvida nas aulas práticas.
O trabalho está organizado nas seguintes componentes:

* Análise Estrutural — implementação de BFS, cálculo de componentes conexas, distribuição de graus e diâmetro da maior componente conexa;

* Métricas de Centralidade — implementação e análise de Degree Centrality, Closeness Centrality, Eigenvector Centrality (via Power Iteration) e Betweenness Centrality (algoritmo de Brandes);

* Análise de Escalabilidade — estudo empírico do comportamento temporal dos algoritmos implementados nos quatro datasets fornecidos, de dimensão variável (desde 187 vértices até 6421 vértices).

Para cada método implementado, é apresentada a análise de complexidade temporal, validação dos resultados e interpretação no contexto narrativo dos universos ficcionais analisados. Os datasets utilizados incluem got_book1.csv, got_full.csv, marvel_small.csv e marvel_full.csv, permitindo comparar o comportamento dos algoritmos em redes de diferentes escalas.




##   Bibliotecas usadas







In [9]:
import numpy as np
import matplotlib.pyplot as plt
import math
import time
import random
from collections import deque
import matplotlib.ticker as ticker
import csv

## TDA Graph

Código adaptado das aulas da semana 7.


In [5]:
class Vertex:
    def __init__(self, vertex_id):
        self._vertex_id = vertex_id

    def __hash__(self):
        return hash(self._vertex_id)

    def __str__(self):
        return 'v{0}'.format(self._vertex_id)

    def __eq__(self, vertex):
        return self._vertex_id == vertex._vertex_id

    def vertex_id(self):
        return self._vertex_id


class Edge:
    def __init__(self, vertex_1, vertex_2, weight):
        self._vertex_1 = vertex_1
        self._vertex_2 = vertex_2
        self._weight = weight

    def __hash__(self):
        return hash((self._vertex_1, self._vertex_2))

    def __str__(self):
        return 'e({0},{1})w={2}'.format(self._vertex_1, self._vertex_2, self._weight)

    def endpoints(self):
        return (self._vertex_1, self._vertex_2)

    def cost(self):
        return self._weight

    def opposite(self, vertex):
        if vertex == self._vertex_1:
            return self._vertex_2
        elif vertex == self._vertex_2:
            return self._vertex_1
        else:
            return None


class Graph:
    def __init__(self):
        self._adjancencies = {}
        self._vertices = {}
        self._n = 0
        self._m = 0

    def __str__(self):
        if self._n == 0:
            return "DAA-Graph: <empty>\n"
        ret = "DAA-Graph:\n"
        for vertex in self._adjancencies.keys():
            ret += str(vertex) + ": "
            for edge in self.incident_edges(vertex.vertex_id()):
                ret += str(edge) + "; "
            ret += "\n"
        return ret

    def order(self):
        return self._n

    def size(self):
        return self._m

    def has_vertex(self, vertex_id):
        return vertex_id in self._vertices

    def has_edge(self, u_id, v_id):
        if not self.has_vertex(u_id) or not self.has_vertex(v_id):
            return False
        vertex_u = self._vertices[u_id]
        vertex_v = self._vertices[v_id]
        return vertex_v in self._adjancencies[vertex_u]

    def insert_vertex(self, vertex_id):
        if not self.has_vertex(vertex_id):
            vertex = Vertex(vertex_id)
            self._vertices[vertex_id] = vertex
            self._adjancencies[vertex] = {}
            self._n += 1

    def insert_edge(self, u_id, v_id, weight=0):
        if not self.has_vertex(u_id):
            self.insert_vertex(u_id)
        if not self.has_vertex(v_id):
            self.insert_vertex(v_id)
        if not self.has_edge(u_id, v_id):
            self._m += 1
        vertex_u = self._vertices[u_id]
        vertex_v = self._vertices[v_id]
        e = Edge(vertex_u, vertex_v, weight)
        self._adjancencies[vertex_u][vertex_v] = e
        self._adjancencies[vertex_v][vertex_u] = e

    def degree(self, vertex_id):
        return len(self._adjancencies[self._vertices[vertex_id]])

    def get_vertex(self, vertex_id):
        return None if not self.has_vertex(vertex_id) else self._vertices[vertex_id]

    def get_edge(self, u_id, v_id):
        if not self.has_edge(u_id, v_id):
            return None
        vertex_u = self._vertices[u_id]
        vertex_v = self._vertices[v_id]
        return self._adjancencies[vertex_u][vertex_v]

    def vertices(self):
        return self._vertices.values()

    def edges(self):
        seen = {}
        for adj_map in self._adjancencies.values():
            for edge in adj_map.values():
                if edge not in seen:
                    yield edge
                seen[edge] = True

    def incident_edges(self, vertex_id):
        vertex = self._vertices[vertex_id]
        for edge in self._adjancencies[vertex].values():
            yield edge

    def remove_vertex(self, vertex_id):
        if self.has_vertex(vertex_id):
            lst_copied = list(self.incident_edges(vertex_id))
            for edge in lst_copied:
                x, y = edge.endpoints()
                self.remove_edge(x.vertex_id(), y.vertex_id())
            del self._adjancencies[self._vertices[vertex_id]]
            del self._vertices[vertex_id]
            self._n -= 1

    def remove_edge(self, u_id, v_id):
        if self.has_edge(u_id, v_id):
            vertex_u = self._vertices[u_id]
            vertex_v = self._vertices[v_id]
            del self._adjancencies[vertex_u][vertex_v]
            if vertex_u != vertex_v:
                del self._adjancencies[vertex_v][vertex_u]
            self._m -= 1

    @staticmethod
    def from_csv(filepath):
        g = Graph()
        with open(filepath, newline='', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                src = row['Source'].strip()
                tgt = row['Target'].strip()
                w   = float(row.get('weight', 1))
                g.insert_edge(src, tgt, w)
        return g

##1. API CentralityAnalyzer

In [ ]:
#API para análise de centralidade de grafos ficcionais.
class CentralityAnalyzer:

    # Recebe um objecto Graph já construído (por Graph.from_csv).
    def __init__(self, graph: Graph):
        self._graph = graph
        self._n = graph.num_vertices()
        self._m = graph.num_edges()


    def bfs(self,source):
        """
        Travessia em largura a partir de `source`.
        Devolve (dist, pred):
          - dist: dict {v -> distância BFS de source a v}
          - pred: dict {v -> predecessor na BFS tree}
        """
        dist = {source: 0}
        pred = {source: None}
        queue = deque([source])
        while queue:
            u = queue.popleft()
            for (v, _) in self._graph.neighbors(u):
                if v not in dist:
                    dist[v] = dist[u] + 1
                    pred[v] = u
                    queue.append(v)
        return dist, pred


    def _path(self, source, target):
        """Devolve a sequência de vértices do caminho mínimo source -> target (BFS)."""
        _, pred = self.bfs(source)
        if target not in pred:
            return None  # sem caminho
        path = []
        v = target
        while v is not None:
            path.append(v)
            v = pred[v]
        path.reverse()
        return path



